<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 2:  Tareas Clásicas de NLP</h1>
    <h2>Traducción Automática (Embeddings)</h2>
</div>


Ahora implementaremos un sistema de traducción automática. Empecemos importando
las funciones requeridas!


```
nltk.download('stopwords')
nltk.download('twitter_samples')
```

In [ ]:
# importar las librerías
import pdb
import pickle
import string
import time
import pandas as pd
import gensim
import matplotlib.pyplot as plt
import nltk
import numpy as np
import scipy
import sklearn
from gensim.models import KeyedVectors
from nltk.corpus import stopwords, twitter_samples
from nltk.tokenize import TweetTokenizer
from os import getcwd


# Embeddings para palabras en Inglés y Frances

Escribiremos un programa que traduzca del inglés al francés.

## Los datos

El conjunto de datos completo para las wordembeddings en inglés es de aproximadamente 3,64 gigabytes, y el francés
son de aproximadamente 629 megabytes. Trabajaremos con un conjunto más pequeño de datos, una muestra significativa.


* English embeddings from Google code archive word2vec
[look for GoogleNews-vectors-negative300.bin.gz](https://code.google.com/archive/p/word2vec/)
    * You'll need to unzip the file first.
* and the French embeddings from
[cross_lingual_text_classification](https://github.com/vjstark/crosslingual_text_classification).
    * in the terminal, type (in one line)
    `curl -o ./wiki.multi.fr.vec https://dl.fbaipublicfiles.com/arrival/vectors/wiki.multi.fr.vec`


#### Subconjunto de los datos




In [ ]:
en_embeddings_subset = pickle.load(open("Data/en_embeddings.p", "rb"))
fr_embeddings_subset = pickle.load(open("Data/fr_embeddings.p", "rb"))

#### Cargue dos diccionarios que mapean las palabras del inglés al francés
* Un diccionario de entrenamiento
* y un diccionario de prueba.

In [ ]:
def get_dict(file_name):
    my_file = pd.read_csv(file_name, delimiter = ' ')
    etof = {}
    for i in range(len(my_file)):
        en = my_file.iloc[i,0]
        fr = my_file.iloc[i,1]
        etof[en] = fr

    return etof

In [ ]:
#Diccionario de Ingles

In [ ]:
#Diccionario de Frances

#### Viendo los diccionarios de Inglés y Francés

* `en_fr_train` es un diccionario donde el key es la palabra en inglés y el valor es la palabra en francés.
```
{'the': 'la',
 'and': 'et',
 'was': 'était',
 'for': 'pour',
```

* `en_fr_test` es similar que `en_fr_train`, pero este es un conjunto de prueba. 

## Generando matrices de embeddings y transformación

####  Traducción de diccionario de inglés a francés mediante embeddings

Implementaremos la función `get_matrices`, que toma los datos cargados y retorna las matrices `X` y `Y`.

Entrada:
- `en_fr` : diccionario del Inglés a Francés
- `en_embeddings` : embeddings de palabras en inglés
- `fr_embeddings` : embeddings de palabras en Francés

Retorna:
- matrices `X` y `Y`, donde cada renglón en X es la palabra embebida para una plabra en inglés, y lo mismo con Y es la palabra embebida en francés.

<div style="width:image width px; font-size:100%; text-align:center;">
<img src='Figures/X_to_Y.jpg' alt="alternate text" width="width" height="height" style="width:800px;height:200px;" /> Figure 2 </div>

Utilice el diccionario `en_fr` para asegurarse de que la i-ésima fila de la matriz` X`
corresponde a la i-ésima fila de la matriz "Y".


In [ ]:
def get_matrices(en_fr, french_vecs, english_vecs):
    
    return X,Y    
        
    

Ahora usaremos la función `get_matrices ()` para obtener los conjuntos `X_train` e` Y_train`
de los embeddings de palabras en inglés y francés.



# Traductores

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Figures/e_to_f.jpg' alt="alternate text" width="width" height="height" style="width:700px;height:200px;" />  </div>

Necesitamos un programa que traduzca palabras del inglés al francés utilizando word embeddings y modelos de espacio vectorial.


##  Traducción como transformación lineal de embeddings
Dados los diccionarios de embeddings de palabras en inglés y francés, crearemos una matriz de transformación `R`
* Dada una embedding en inglés, $ \mathbf {e} $, podemos multiplicar $ \mathbf {eR} $ para obtener una nueva palabra (nuevo embedding) $ \mathbf {f} $.

    
* Después, Calcularemos los vecinos más cercanos a `f` en los embeddings francesas y recomendar la palabra que es más similar a los embeddings de palabras transformadas.

### Describir la traducción como el problema de minimización

Necesitamos encontrar una matriz `R` que minimice la siguiente ecuación.

$$\arg \min _{\mathbf{R}}\| \mathbf{X R} - \mathbf{Y}\|_{F}\tag{1} $$

### Norma de Frobenius

La norma de Frobenius de una matriz $ A $ (asumiendo que es de dimensión $ m, n $) se define como la raíz cuadrada de la suma de los cuadrados absolutos de sus elementos:

$$\|\mathbf{A}\|_{F} \equiv \sqrt{\sum_{i=1}^{m} \sum_{j=1}^{n}\left|a_{i j}\right|^{2}}\tag{2}$$

### Función de perdida

En las aplicaciones del mundo real, la pérdida de la norma Frobenius:

$$\| \mathbf{XR} - \mathbf{Y}\|_{F}$$

a menudo se reemplaza por su valor al cuadrado dividido por $ m $:

$$ \frac{1}{m} \|  \mathbf{X R} - \mathbf{Y} \|_{F}^{2}$$

donde $ m $ es el número de ejemplos (filas en $ \mathbf {X} $).

* Se encuentra la misma R cuando se usa esta función de pérdida en comparación con la norma de Frobenius original.
* La razón para tomar el cuadrado es que es más fácil calcular el gradiente del Frobenius al cuadrado.
* La razón para dividir entre $ m $ es que estamos más interesados en la pérdida promedio por inserción que en la pérdida de todo el conjunto de entrenamiento.
     * La pérdida de todo el conjunto de entrenamiento aumenta con más palabras (ejemplos de entrenamiento),
     por lo que tomar el promedio nos ayuda a rastrear la pérdida promedio independientemente del tamaño del conjunto de entrenamiento.



### Implementación del mecanismo de traducción

#### Calculando el loss
* La función de pérdida será la norma de Frobenoius al cuadrado de la diferencia entre
matriz y su aproximación, dividida por el número de ejemplos de entrenamiento $ m $.
* Su fórmula es:
$$ L(X, Y, R)=\frac{1}{m}\sum_{i=1}^{m} \sum_{j=1}^{n}\left( a_{i j} \right)^{2}$$

donde $a_{i j}$ es el valo de la $i$-ésimo renglón y $j$-ésima columna de la matriz $\mathbf{XR}-\mathbf{Y}$.

* Calcular la aproximación de `Y` mediante la matriz multiplicando` X` y `R`
* Calcular la diferencia `XR - Y`
* Calcular la norma de Frobenius al cuadrado de la diferencia y divídala por $ m $.

In [ ]:
def compute_loss(X,Y,R):
    
    return loss
    


### Calculando el gradiente de la función loss con respecto a la matriz de transforación R

* Calcular el gradiente de la pérdida con respecto a la matriz de transformación "R".
* El gradiente nos da la dirección en la que debemos disminuir `R`
para minimizar la pérdida.
* $ m $ es el número de ejemplos de entrenamiento (número de filas en $ X $).
* La fórmula para el gradiente de la función de pérdida $ 𝐿 (𝑋, 𝑌, 𝑅) $ es:

$$\frac{d}{dR}𝐿(𝑋,𝑌,𝑅)=\frac{d}{dR}\Big(\frac{1}{m}\| X R -Y\|_{F}^{2}\Big) = \frac{2}{m}X^{T} (X R - Y)$$



In [ ]:
def compute_gradient(X,Y,R):
     
    return gradient


### Encontrar la R óptima con el algoritmo de descenso de gradiente

#### Gradient descent


Pseudocódigo:
1. Calcular el gradiente $g$ del loss con respecto a la matriz $R$.
2. Update $R$ con la formula:
$$R_{\text{new}}= R_{\text{old}}-\alpha g$$

Donde $\alpha$ es el learning rate, que es un escalar.

#### Learning rate

* La tasa de aprendizaje o "tamaño de paso" $ \ alpha $ es un coeficiente que decide cuánto queremos cambiar $ R $ en cada paso.
* Si cambiamos $ R $ demasiado, podríamos saltarnos el óptimo dando un paso demasiado grande.
* Si solo hacemos pequeños cambios en $ R $, necesitaremos muchos pasos para alcanzar el óptimo.
* La tasa de aprendizaje $ \ alpha $ se usa para controlar esos cambios.
* Los valores de $ \ alpha $ se eligen dependiendo del problema, y usaremos `learning_rate` $ = 0.0003 $ como valor predeterminado para nuestro algoritmo.

In [ ]:
def align_embeddings(X,Y, train_steps = 100, learning_rate = 0.0003):

    return R
    

In [ ]:
np.random.seed(129)
m=10
n=5
X = np.random.rand(m,n)
Y = np.random.rand(m,n)*.1
#Probar la función


## Calcular la matriz de transformación R


loss at iteration 191 is: 0.750715865735653
loss at iteration 192 is: 0.7461752199674594
loss at iteration 193 is: 0.741756490574503
loss at iteration 194 is: 0.7374560733570299
loss at iteration 195 is: 0.7332704815954062
loss at iteration 196 is: 0.7291963418216838
loss at iteration 197 is: 0.7252303897591594
loss at iteration 198 is: 0.721369466422584
loss at iteration 199 is: 0.7176105143720448
loss at iteration 200 is: 0.7139505741138531
loss at iteration 201 is: 0.7103867806421019
loss at iteration 202 is: 0.7069163601148487
loss at iteration 203 is: 0.7035366266591626
loss at iteration 204 is: 0.7002449792995558
loss at iteration 205 is: 0.6970388990045426
loss at iteration 206 is: 0.6939159458463544
loss at iteration 207 is: 0.6908737562690396
loss at iteration 208 is: 0.6879100404603988
loss at iteration 209 is: 0.6850225798234384
loss at iteration 210 is: 0.6822092245431776
loss at iteration 211 is: 0.6794678912448879
loss at iteration 212 is: 0.6767965607399775
loss at itera

loss at iteration 377 is: 0.5686167334411015
loss at iteration 378 is: 0.5685466826988004
loss at iteration 379 is: 0.5684779546491274
loss at iteration 380 is: 0.5684105226586321
loss at iteration 381 is: 0.5683443606688969
loss at iteration 382 is: 0.5682794431832174
loss at iteration 383 is: 0.5682157452536102
loss at iteration 384 is: 0.5681532424681442
loss at iteration 385 is: 0.56809191093858
loss at iteration 386 is: 0.5680317272883223
loss at iteration 387 is: 0.5679726686406622
loss at iteration 388 is: 0.5679147126073114
loss at iteration 389 is: 0.567857837277216
loss at iteration 390 is: 0.5678020212056437
loss at iteration 391 is: 0.5677472434035448
loss at iteration 392 is: 0.5676934833271623
loss at iteration 393 is: 0.5676407208679026
loss at iteration 394 is: 0.5675889363424506
loss at iteration 395 is: 0.5675381104831243
loss at iteration 396 is: 0.5674882244284666
loss at iteration 397 is: 0.5674392597140602
loss at iteration 398 is: 0.5673911982635673
loss at itera


## Probando el traductor

### Algoritmo k-Nearest neighbors 

[k-Nearest neighbors algorithm](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) 
* k-NN es un método que toma un vector como entrada y encuentra los otros vectores en el conjunto de datos que están más cerca de él.
* La 'k' es el número de "vecinos más cercanos" a encontrar (por ejemplo, k = 2 encuentra los dos vecinos más cercanos).

### Buscando el embedding de la traducción 
Dado que estamos aproximando la función de traducción de los embeddings de inglés a francés mediante una matriz de transformación lineal $ \mathbf {R} $, la mayoría de las veces no obtendremos lel embedding de una palabra francesa cuando transformamos los embeddings $ \mathbf { e} $ de alguna palabra en inglés en particular en el espacio de embeddings francés.
* ¡Aquí es donde $ k $ -NN se vuelve realmente útil! Al usar $ 1 $ -NN con $ \mathbf {eR} $ como entrada, podemos buscar un embedding $ \mathbf {f} $ (como una fila) en la matriz $ \mathbf {Y} $ que es la más cercana a el vector transformado $ \mathbf {eR} $

### Similaridad por  Coseno
Similitud de coseno entre los vectores $ u $ y $ v $ calculada como el coseno del ángulo entre ellos.
La formula es
$$\cos(u,v)=\frac{u\cdot v}{\left\|u\right\|\left\|v\right\|}$$

* $\cos(u,v)$ = $1$ cuando $u$ y $v$ se encuentran en la misma línea y tienen la misma dirección.
* $\cos(u,v)$ es $-1$ Cuando ellas tienen direcciones exactamente opuestas.
* $\cos(u,v)$ es $0$ cuando los vectores son ortogonales (perpendiculares) entre sí.

#### Nota: La distancia y la similitud son cosas bastante opuestas.
* Podemos obtener la métrica de la distancia a partir de la similitud del coseno, pero la similitud del coseno no se puede usar directamente como la métrica de la distancia.
* Cuando la similitud del coseno aumenta (hacia $ 1 $), la "distancia" entre los dos vectores disminuye (hacia $ 0 $).
* Podemos definir la distancia del coseno entre $ u $ y $ v $ como
$$ d_{\text {cos}} (u, v) = 1- \cos(u, v) $$

In [ ]:
def cosine_similarity(A,B):
    dot = np.dot(A, B)
    norma = np.linalg.norm(A)
    normb = np.linalg.norm(B)
    cos = dot / (norma * normb)
    return cos

Completando la función `nearest_neighbor()`:



In [ ]:
def nearest_neighbor(v, candidates, k=1):
    
    return k_idx, similarity_l
        


In [ ]:
#Probando l función
v= np.array([1,0,1])
candidates=np.array([[1,0,5],
                     [-2,5,3],
                     [2,0,1],
                     [6,-9,5],
                     [9,9,9]])

kidx, sim = nearest_neighbor(v,candidates,2)


### Probando la traducción y calculando el accuracy del modelo

* Calculando el accuracy como $$\text{accuracy}=\frac{\#(\text{correct predictions})}{\#(\text{total predictions})}$$

In [ ]:
def test_vocabulary(X,Y,R):
    
    return accuracy

        
    

In [ ]:
#Probando el modelo


In [ ]:
#Calculando el accuracy


In [ ]:
#Traducir laa palabra en frances en función a una palbra en inglés
def predict_fr_word(en_word, en_embedding, R):

    return fr_word